# 1 - Simulación de datos sintéticos (FinanceAI)

**1. Simulación** > 2. EDA > 3. Entrenamiento

Este cuaderno genera un conjunto de datos sintéticos autónomo, limpio y **estructuralmente coherente**. En lugar de asignar valores aleatorios ciegos, el perfil financiero de cada usuario (endeudamiento y ahorro) se calcula matemáticamente evaluando su historial anual de transacciones generadas. Esto garantiza una correlación lógica estricta para el entrenamiento de los algoritmos de Machine Learning, replicando el funcionamiento de un entorno de negocio real.

In [1]:
import pandas as pd
import numpy as np
import os
import json
from faker import Faker

# Fijar semilla de aleatoriedad para garantizar reproducibilidad total en el proyecto
SEED = 42
np.random.seed(SEED)
Faker.seed(SEED)
fake = Faker('es_ES')

## 1. Generación de usuarios
1800 usuarios con sus ingresos y datos de identidad base.

In [19]:
n_usuarios = 1800

# Probando Faker para usar nombres realistas y únicos
nombres = [fake.first_name() for _ in range(n_usuarios)]

# Generación de ingresos mensuales ($1500 a $6000) con random.uniform
ingresos = np.round(np.random.uniform(1500, 5000, size=n_usuarios), 2)

df_usuarios = pd.DataFrame({
    'id': range(1, n_usuarios + 1),
    'nombre': nombres,
    'ingreso_mensual': ingresos
})

print(f"check: {df_usuarios.tail()}")
print("- - -")
print("Estructura generada (instances, atributo):", df_usuarios.shape)

check:         id  nombre  ingreso_mensual
1795  1796  Evelia          3405.68
1796  1797  Elodia          2766.85
1797  1798    Rafa          3551.47
1798  1799  Moreno          4456.54
1799  1800   Celso          4564.06
- - -
Estructura generada (instances, atributo): (1800, 3)


## 2. Generación de transacciones
240k consumos asociados a los usuarios y limitados estrictamente a las 10 categorías definitivas.

In [3]:
n_transacciones = 240000

# 10 categorías acordadas + descripciones random
diccionario_conceptos = {
    'Alimentacion': ['supermercado coto', 'verduleria el sol', 'carniceria central', 'almacen san martin', 'compra panaderia', 'supermercado carrefour', 'compras fiambreria', 'compra dia'],
    'Educacion': ['cuota universidad', 'compra libros', 'curso de programacion', 'matricula colegio', 'utiles escolares', 'taller ingles', 'cuota jardin'],
    'Electrodomesticos': ['compra heladera', 'lavarropas fravega', 'televisor garbarino', 'microondas musimundo', 'licuadora philips', 'pava electrica', 'aire acondicionado'],
    'Inversion': ['compra dolares', 'fondo comun inversion', 'plazo fijo', 'cedears', 'bonos del estado', 'acciones ypf', 'transferencia broker'],
    'Ocio': ['salida cine', 'suscripcion netflix', 'cena restaurante', 'entradas recital', 'cerveceria', 'suscripcion spotify', 'juegos steam', 'cafeteria'],
    'Salud': ['estudios clinicos', 'compra farmacia', 'consulta medica', 'cuota prepaga', 'medicamentos', 'dentista', 'analisis sangre', 'optica'],
    'Servicios': ['factura luz edesur', 'abono internet', 'servicio agua', 'factura gas', 'telefonia movil', 'impuesto municipal', 'abl', 'rentas'],
    'Transporte': ['viaje uber', 'colectivo', 'carga sube', 'combustible ypf', 'peaje autopista', 'taxi', 'viaje cabify', 'combustible shell'],
    'Vestimenta': ['compra zapatillas', 'pantalon jean', 'remera algodon', 'campera invierno', 'ropa deportiva', 'local indumentaria', 'zapatos'],
    'Vivienda': ['pago alquiler', 'expensas edificio', 'servicio plomeria', 'ferreteria', 'pintura habitacion', 'reparacion electrica', 'materiales construccion']
}

# Limites (base) de montos realistas - sujeto a cambiar montos a la brevedad
rangos_montos = {
    'Alimentacion': (15, 100), 'Educacion': (30, 200), 'Electrodomesticos': (200, 400), 'Inversion': (100, 300),
    'Ocio': (20, 80), 'Salud': (40, 200), 'Servicios': (20, 150), 'Transporte': (3, 200),
    'Vestimenta': (10, 300), 'Vivienda': (200, 600)
}

categorias = list(diccionario_conceptos.keys()) # Seteando las categorías
usuario_ids = np.random.randint(1, n_usuarios + 1, size=n_transacciones) 

# Asignando categorias simulando comportamientos de gasto frecuentes
prob_categorias = [0.15, 0.05, 0.05, 0.10, 0.10, 0.05, 0.20, 0.10, 0.05, 0.15] # Se busca respetar el orden alfabético de categorías
# Clasifica (sortea) el lote de transacciones. Utiliza un set de probabilidades calibrado para intentar emular frecuencias reales, 
# asegurando que ir al supermercado salga sorteado muchas más veces que comprar un electrodoméstico.
selected_categories = np.random.choice(categorias, size=n_transacciones, p=prob_categorias) 

fechas_random = pd.to_datetime('2026-01-01') + pd.to_timedelta(np.random.randint(0, 365, size=n_transacciones), unit='d') # Repartiendo fechas random a las transacciones. Cuidado con años bisiestos

descripciones = []
montos = []

# Novedad: Diccionario rápido para conocer el índice de poder adquisitivo de cada usuario
# Base: 1500 dólares de sueldo.
# Funcionalidad sugerida para ahorrar RAM y ahorrar tiempo al código
dict_indices = {row['id']: (row['ingreso_mensual'] / 1500) for _, row in df_usuarios.iterrows()} # Itera usuario por usuario, tomando el sueldo minimo como denominador para repartir multiplicadores float usando el id usuario como clave (clave:valor) 
categorias_elasticas = ['Vivienda', 'Educacion', 'Inversion', 'Ocio', 'Vestimenta', 'Electrodomesticos'] # definición de categorías elásticas

# PRUEBA - Crear tickets de compra. Tiene en cuenta el poder adquisitivo y las categorías elásticas
for user_id, cat in zip(usuario_ids, selected_categories):
    desc = np.random.choice(diccionario_conceptos[cat])
    
    # Inyección de ruido en texto (simulando errores de usuario o teclado)
    if False: # np.random.rand() < 0.20:
        prefijos = ["pago ", "compra ", "tarjeta ", "fac ", ""]
        desc = str(np.random.choice(prefijos)) + desc
    if False: # np.random.rand() < 0.10:
        desc = desc.replace("a", "q", 1) if "a" in desc else desc.replace("e", "w", 1)

    min_m, max_m = rangos_montos[cat]
    idx_adquisitivo = dict_indices[user_id]
    
    # Factor de escalado dinámico según sueldo
    if cat in categorias_elasticas:
        max_m = max_m * (1 + (idx_adquisitivo - 1) * 0.5)
        min_m = min_m * (1 + (idx_adquisitivo - 1) * 0.2) # el minimo sube, pero más suave
    else:
        # Inelásticas (crecen muy poco)
        max_m = max_m * (1 + (idx_adquisitivo - 1) * 0.2)
        
    # --- Distribución Triangular (Mejora Estadística) ---
    if cat in ['Servicios', 'Educacion', 'Vivienda']:
        moda_m = min_m + (max_m - min_m) * 0.5
    else:
        moda_m = min_m + (max_m - min_m) * 0.25
    monto = round(np.random.triangular(min_m, moda_m, max_m), 2)
    descripciones.append(desc)
    montos.append(monto)



# 10% de ruido de etiquetado en categorías para evitar que el algoritmo NLP sea 100% perfecto - DESACTIVADO
ruido_mask = np.random.rand(n_transacciones) < 0.0 # 0.10
categorias_ruido = np.random.choice(categorias, size=ruido_mask.sum())
selected_categories[ruido_mask] = categorias_ruido

df_transacciones = pd.DataFrame({
    'id': range(1, n_transacciones + 1),
    'usuario_id': usuario_ids,
    'descripcion': descripciones,
    'valor': montos,
    'categoria': selected_categories,
    'fecha': fechas_random.strftime('%Y-%m-%d')
})

print("Transacciones simuladas:", df_transacciones.shape)


Transacciones simuladas: (240000, 6)


### 2.1 Revisión de transacciones


In [23]:
# Para ver cuántas transacciones promedio tiene cada usuario
tx_por_usuario = df_transacciones.groupby('usuario_id').size().reset_index(name='tx_anuales')
tx_por_usuario['tx_mensuales'] = tx_por_usuario['tx_anuales'] / 12.0

print("Transacciones por Usuario:")
display(tx_por_usuario[['tx_anuales', 'tx_mensuales']].mean())

Transacciones por Usuario:


tx_anuales      133.333333
tx_mensuales     11.111111
dtype: float64

## 3. Perfil Financiero - Revisar umbrales
Auditoría de 4 pasos sin bucles

`La Regla Bancaria Internacional (28/36). Se eligieron umbrales "bajos y apretados" (22% y 26%) para forzar a que la distribución poblacional quedara repartida aproximadamente en un 40/35/25, permitiendo un entrenamiento de Machine Learning estable y estadísticamente sano.`

##### **Aislar los Gastos Fijos (Vivienda y Servicios):** El algoritmo busca todos los pagos de alquiler, luz o agua de un usuario y saca el promedio de cuánto paga al mes. Supongamos que el usuario "Juan" paga un promedio de $600 mensuales por estos conceptos.

##### **Ratio de Endeudamiento:** Toma esos $600 y los divide por su sueldo. El cálculo arroja un 40%. Eso significa que Juan compromete el 40% de su sueldo solo para sobrevivir mes a mes.

##### **Calcular el Hábito de Ahorro:** El algoritmo cuenta cuántos "tickets de compra" de Juan pertenecen a la categoría "Inversión" a lo largo del año. Si Juan tiene cero tickets de inversión, se le cataloga con Frecuencia de Ahorro "Ninguna".

##### *Armado del perfil financiero:** Cruza el endeudamiento con el ahorro de la siguiente manera: 
> * Si el endeudamiento de la persona es mayor al 26% de su sueldo, automáticamente se etiqueta como 'En Riesgo' (la persona está ahogada financieramente).
> * Si su endeudamiento es bajo (menor al 22%) Y además tiene un hábito de ahorro Medio o Alto, se corona como 'Saludable'.
> * Cualquier combinación intermedia o gris (ejemplo, tiene poco endeudamiento pero nunca invierte un centavo), cae en la categoría 'En Observación'.



In [5]:
# 1. Promedio mensual de gastos fijos (Vivienda y Servicios)
gastos_fijos = df_transacciones[df_transacciones['categoria'].isin(['Vivienda', 'Servicios'])]
gastos_fijos_promedio = gastos_fijos.groupby(['usuario_id', 'categoria'])['valor'].mean().unstack(fill_value=0)
if 'Vivienda' not in gastos_fijos_promedio.columns: gastos_fijos_promedio['Vivienda'] = 0
if 'Servicios' not in gastos_fijos_promedio.columns: gastos_fijos_promedio['Servicios'] = 0
gastos_fijos_promedio['gasto_fijo_total'] = gastos_fijos_promedio['Vivienda'] + gastos_fijos_promedio['Servicios']

# 2. Conteo de inversiones
inversiones = df_transacciones[df_transacciones['categoria'] == 'Inversion']
conteo_inversiones = inversiones.groupby('usuario_id').size().reset_index(name='n_inversion_anual')

# 3. Merge con usuarios
df_calc = df_usuarios[['id', 'ingreso_mensual']].copy()
df_calc = df_calc.merge(gastos_fijos_promedio[['gasto_fijo_total']], left_on='id', right_on='usuario_id', how='left').fillna(0)
df_calc = df_calc.merge(conteo_inversiones, left_on='id', right_on='usuario_id', how='left').fillna(0)

# 4. Cálculos vectorizados
df_calc['nivel_endeudamiento'] = np.round((df_calc['gasto_fijo_total'] / df_calc['ingreso_mensual']) * 100, 2)
df_calc['nivel_endeudamiento'] = df_calc['nivel_endeudamiento'].clip(5.0, 90.0)

df_calc['inversiones_mensuales'] = df_calc['n_inversion_anual'] / 12.0

# Asignar frecuencia de ahorro
condiciones_ahorro = [
    df_calc['inversiones_mensuales'] == 0,
    df_calc['inversiones_mensuales'] < 1.0,
    df_calc['inversiones_mensuales'] < 3.0
]
opciones_ahorro = ['Ninguna', 'Baja', 'Media']
df_calc['frecuencia_ahorro'] = np.select(condiciones_ahorro, opciones_ahorro, default='Alta')

# Asignar perfil financiero
condiciones_perfil = [
    df_calc['nivel_endeudamiento'] > 26.0,
    (df_calc['nivel_endeudamiento'] > 22.0) | (df_calc['frecuencia_ahorro'] == 'Ninguna'),
    (df_calc['nivel_endeudamiento'] <= 22.0) & (df_calc['frecuencia_ahorro'].isin(['Media', 'Alta']))
]
opciones_perfil = ['En riesgo', 'En observacion', 'Saludable']
df_calc['perfil_financiero'] = np.select(condiciones_perfil, opciones_perfil, default='En observacion')

# Integrar resultados al dataframe original
df_usuarios = pd.merge(df_usuarios, df_calc[['id', 'nivel_endeudamiento', 'frecuencia_ahorro', 'perfil_financiero']], on='id')

print("Perfiles consistentes calculados. Distribución:")
print(df_usuarios['perfil_financiero'].value_counts(normalize=True) * 100)


Perfiles consistentes calculados. Distribución:
perfil_financiero
En observacion    41.333333
Saludable         36.833333
En riesgo         21.833333
Name: proportion, dtype: float64


## 4. Prevención de Data Leakage - Prueba
Prevención de fugas:
*   **Para Perfiles (Usuarios):** Corte *Cross-Sectional*.
*   **Para Transacciones (NLP):** Corte temporal. El modelo entrena con el historial de enero-agosto y se evalúa de manera ciega sobre los gastos del futuro (Nov-Dic).

In [6]:
# 1. Split Transversal (Group K-Fold) para Usuarios
splits_usr = np.random.choice(['train', 'val', 'test'], size=n_usuarios, p=[0.6, 0.2, 0.2])
df_usuarios['split'] = splits_usr

# 2. Split Temporal (Out-of-Time) para Transacciones
# Enero a Agosto (train), Sept-Oct (val), Nov-Dic (test)
meses = pd.to_datetime(df_transacciones['fecha']).dt.month
condiciones = [
    meses <= 8,
    meses.isin([9, 10]),
    meses >= 11
]
df_transacciones['split'] = np.select(condiciones, ['train', 'val', 'test'])

print("Distribución Transversal de Usuarios (Cross-sectional):")
print(df_usuarios['split'].value_counts(normalize=True) * 100)
print("\nDistribución Temporal de Transacciones (Out-of-Time):")
print(df_transacciones['split'].value_counts(normalize=True) * 100)

Distribución Transversal de Usuarios (Cross-sectional):
split
train    59.222222
val      20.555556
test     20.222222
Name: proportion, dtype: float64

Distribución Temporal de Transacciones (Out-of-Time):
split
train    66.45875
val      16.79250
test     16.74875
Name: proportion, dtype: float64


## 5. Exportación de Archivos Semilla
Exportamos tablas robustas y finalizadas a `data/`.

In [7]:
os.makedirs('data', exist_ok=True)

# 1. Exportación Cruda (Uso Interno de Data Science)
# Estos archivos mantienen la columna "split" esencial para EDA y Modelado.
df_usuarios.to_csv('data/usuarios.csv', index=False)
df_transacciones.to_csv('data/transacciones.csv', index=False)
df_usuarios.to_json('data/usuarios.json', orient='records', indent=2, force_ascii=False)
df_transacciones.to_json('data/transacciones.json', orient='records', indent=2, force_ascii=False)

# 2. Exportación Limpia (Uso Externo para Backend)
# Se remueve "split" para evitar problemas de compatibilidad en Java.
df_usuarios.drop(columns=['split']).to_csv('data/usuarios_backend.csv', index=False)
df_transacciones.drop(columns=['split']).to_csv('data/transacciones_backend.csv', index=False)
df_usuarios.drop(columns=['split']).to_json('data/usuarios_backend.json', orient='records', indent=2, force_ascii=False)
df_transacciones.drop(columns=['split']).to_json('data/transacciones_backend.json', orient='records', indent=2, force_ascii=False)

print("¡Exportación exitosa! Semillas crudas y limpias generadas correctamente.")


¡Exportación exitosa! Semillas crudas y limpias generadas correctamente.


In [8]:
# check rápido
df_usuarios.head()

,id,nombre,ingreso_mensual,nivel_endeudamiento,frecuencia_ahorro,perfil_financiero,split
0,1,Feliciana,2810.89,22.18,Baja,En observacion,train
1,2,Dani,4827.50,19.42,Media,Saludable,test
2,3,Amador,4061.98,19.05,Media,Saludable,test
3,4,Manuela,3595.30,19.69,Media,Saludable,train
4,5,Leonardo,2046.07,26.09,Baja,En riesgo,train
